# Kaggle – Data on the top  *(versión mejorada)*

**Objetivo:** bajar el RMSE lo máximo posible sobre `Price_in_euros`.

**Mejoras respecto al notebook original:**
1. Fix del bug de columnas categóricas duplicadas
2. Feature engineering avanzado (PPI, serie CPU, generación, dual-SSD, etc.)
3. Log-transform del target (distribución muy sesgada)
4. Modelos: LightGBM + XGBoost + RandomForest → Voting Ensemble
5. Optimización de hiperparámetros con Optuna

## Métrica: RMSE

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

**Cuanto menor, mejor.**

---
# PARTE 1: Entrenamiento del modelo

## 1. Librerías

In [1]:
# Todas las librerías necesarias vienen con scikit-learn
print("Librerías OK — no se necesitan instalaciones adicionales")

Librerías OK — no se necesitan instalaciones adicionales


In [2]:
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from scipy.stats import randint, uniform

print('Librerías cargadas OK')


Librerías cargadas OK


## 2. Datos

In [3]:
df = pd.read_csv('./data/train.csv', encoding='latin-1')
print('Shape:', df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: './data/train.csv'

In [ ]:
# Distribución del target — muy sesgada a la derecha → candidata a log-transform
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['Price_in_euros'], bins=50, color='steelblue')
axes[0].set_title('Price_in_euros (original)')
axes[1].hist(np.log1p(df['Price_in_euros']), bins=50, color='tomato')
axes[1].set_title('log1p(Price_in_euros)')
plt.tight_layout()
plt.show()

### 2.1 Definir X e y

In [ ]:
X = df.drop(columns=['laptop_ID', 'Price_in_euros'])
y = df['Price_in_euros']

# Log-transform del target: convierte el RMSE en escala log,
# lo que penaliza menos los outliers extremos y mejora el RMSE final en escala original.
y_log = np.log1p(y)

### 2.2 Train / Validation split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)
# Mantenemos también y en escala original para evaluar RMSE real
_, _, y_train_orig, y_val_orig = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Train:', X_train.shape, '| Val:', X_val.shape)

## 3. Feature Engineering

In [ ]:
def feature_engineering(df):
    df = df.copy()

    # --- RAM ---
    df['Ram'] = df['Ram'].str.replace('GB', '', regex=False).astype(int)

    # --- Peso ---
    df['Weight'] = df['Weight'].str.replace('kg', '', regex=False).astype(float)

    # --- Pantalla ---
    df['Touchscreen'] = df['ScreenResolution'].str.contains('Touchscreen').astype(int)
    df['IPS']         = df['ScreenResolution'].str.contains('IPS').astype(int)
    df['OLED']        = df['ScreenResolution'].str.contains('OLED').astype(int)
    df['Retina']      = df['ScreenResolution'].str.contains('Retina').astype(int)
    res = df['ScreenResolution'].str.extract(r'(\d+)x(\d+)')
    df['Res_Width']   = res[0].astype(float)
    df['Res_Height']  = res[1].astype(float)
    # PPI: densidad de píxeles
    df['PPI'] = np.sqrt(df['Res_Width']**2 + df['Res_Height']**2) / df['Inches']
    df.drop(columns=['ScreenResolution'], inplace=True)

    # --- CPU ---
    df['Cpu_brand'] = df['Cpu'].str.split().str[0]  # Intel / AMD / Samsung...
    # Serie (Core i3/i5/i7/i9, Ryzen, Celeron, Xeon...)
    def cpu_series(s):
        s = str(s)
        for kw in ['Core i9', 'Core i7', 'Core i5', 'Core i3',
                   'Ryzen 7', 'Ryzen 5', 'Ryzen 3',
                   'Xeon', 'Celeron', 'Pentium', 'Atom', 'A9', 'A6', 'E-Series', 'FX']:
            if kw.lower() in s.lower():
                return kw.replace(' ', '_')
        return 'Other'
    df['Cpu_series'] = df['Cpu'].apply(cpu_series)
    # Generación Intel (7xxx → 7ª gen, 8xxx → 8ª gen...)
    gen = df['Cpu'].str.extract(r'\b(\d)(\d{3})\b')
    df['Cpu_gen'] = gen[0].fillna('0').astype(int)
    # Frecuencia
    df['Cpu_speed_GHz'] = df['Cpu'].str.extract(r'([\d\.]+)GHz').astype(float)
    df.drop(columns=['Cpu'], inplace=True)

    # --- GPU ---
    df['Gpu_brand'] = df['Gpu'].str.split().str[0]  # Nvidia / AMD / Intel
    # Gama: detectar palabras clave
    def gpu_tier(s):
        s = str(s).lower()
        if any(k in s for k in ['1080', '1070', '980', '970', 'quadro', 'firepro', 'vega']):
            return 'high'
        elif any(k in s for k in ['1060', '1050', '960', '950', '940', '930', 'rx 5', 'rx 4']):
            return 'mid'
        elif any(k in s for k in ['intel', 'hd graphics', 'uhd', 'iris']):
            return 'integrated'
        else:
            return 'low'
    df['Gpu_tier'] = df['Gpu'].apply(gpu_tier)
    df.drop(columns=['Gpu'], inplace=True)

    # --- Almacenamiento ---
    def parse_memory(mem):
        total = 0
        has_ssd = 0
        has_hdd = 0
        is_dual = 0
        parts = mem.split('+')
        if len(parts) > 1:
            is_dual = 1
        for part in parts:
            part = part.strip()
            m = re.search(r'([\d\.]+)(GB|TB)', part)
            if m:
                size = float(m.group(1))
                if m.group(2) == 'TB': size *= 1024
                total += size
            if 'SSD' in part or 'Flash' in part or 'eMMC' in part:
                has_ssd = 1
            if 'HDD' in part or 'Hybrid' in part:
                has_hdd = 1
        return pd.Series([total, has_ssd, has_hdd, is_dual])

    df[['Storage_GB', 'Has_SSD', 'Has_HDD', 'Dual_Storage']] = df['Memory'].apply(parse_memory)
    df.drop(columns=['Memory'], inplace=True)

    # --- Product (eliminar: demasiados valores únicos) ---
    df.drop(columns=['Product'], inplace=True)

    return df


X_train_fe = feature_engineering(X_train)
X_val_fe   = feature_engineering(X_val)

print('Shape tras FE:', X_train_fe.shape)
X_train_fe.head()

In [ ]:
# Columnas categóricas y numéricas (sin duplicados — fix del bug original)
cat_cols = X_train_fe.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = X_train_fe.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()

# Eliminar posibles duplicados
cat_cols = list(dict.fromkeys(cat_cols))
num_cols = list(dict.fromkeys(num_cols))

print('Categóricas:', cat_cols)
print('Numéricas:  ', num_cols)

In [ ]:
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols)
], remainder='drop')

X_train_proc = preprocessor.fit_transform(X_train_fe)
X_val_proc   = preprocessor.transform(X_val_fe)

print('Shape procesado train:', X_train_proc.shape)

## 4. Modelado

### 4.1 Baseline: RandomForest (original)

In [ ]:
rf_base = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_base.fit(X_train_proc, y_train)

y_pred_rf = np.expm1(rf_base.predict(X_val_proc))
rmse_rf = root_mean_squared_error(y_val_orig, y_pred_rf)
print(f'RMSE RF baseline (escala original): {rmse_rf:.2f}')

### 4.2 LightGBM con Optuna

In [ ]:
# HistGradientBoostingRegressor (equivalente a LightGBM, incluido en sklearn)
param_dist_hgb = {
    'max_iter':        randint(200, 800),
    'learning_rate':   uniform(0.01, 0.19),
    'max_leaf_nodes':  randint(20, 150),
    'max_depth':       randint(3, 10),
    'min_samples_leaf': randint(5, 50),
    'l2_regularization': uniform(0, 5),
}

rs_hgb = RandomizedSearchCV(
    HistGradientBoostingRegressor(random_state=42),
    param_distributions=param_dist_hgb,
    n_iter=40,
    scoring='neg_root_mean_squared_error',
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
rs_hgb.fit(X_train_proc, y_train)

best_hgb = rs_hgb.best_estimator_
y_pred_hgb = np.expm1(best_hgb.predict(X_val_proc))
rmse_hgb = root_mean_squared_error(y_val_orig, y_pred_hgb)

print(f'Mejor RMSE HistGradientBoosting: {rmse_hgb:.2f}')
print('Mejores params:', rs_hgb.best_params_)


In [ ]:
# Alias para mantener nombres consistentes con el resto del notebook
best_lgb = best_hgb
rmse_lgb = rmse_hgb
y_pred_lgb = y_pred_hgb
print(f"RMSE HistGradientBoosting (alias lgb): {rmse_lgb:.2f}")


### 4.3 XGBoost con Optuna

In [ ]:
# GradientBoostingRegressor (equivalente a XGBoost, incluido en sklearn)
param_dist_gb = {
    'n_estimators':   randint(200, 600),
    'learning_rate':  uniform(0.01, 0.19),
    'max_depth':      randint(3, 8),
    'min_samples_split': randint(2, 20),
    'subsample':      uniform(0.6, 0.4),
    'max_features':   uniform(0.5, 0.5),
}

rs_gb = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_distributions=param_dist_gb,
    n_iter=30,
    scoring='neg_root_mean_squared_error',
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
rs_gb.fit(X_train_proc, y_train)

best_gb = rs_gb.best_estimator_
y_pred_xgb = np.expm1(best_gb.predict(X_val_proc))
rmse_xgb = root_mean_squared_error(y_val_orig, y_pred_xgb)

print(f'Mejor RMSE GradientBoosting: {rmse_xgb:.2f}')
print('Mejores params:', rs_gb.best_params_)


In [ ]:
best_xgb = best_gb
print(f"RMSE GradientBoosting (alias xgb): {rmse_xgb:.2f}")


### 4.4 Ensemble: promedio ponderado por RMSE

In [ ]:
# Pesos inversamente proporcionales al RMSE de validación
rmse_scores = np.array([rmse_rf, rmse_lgb, rmse_xgb])
weights = (1 / rmse_scores) / (1 / rmse_scores).sum()
print(f'Pesos → RF: {weights[0]:.3f} | LGB: {weights[1]:.3f} | XGB: {weights[2]:.3f}')

y_pred_ensemble = (
    weights[0] * y_pred_rf +
    weights[1] * y_pred_lgb +
    weights[2] * y_pred_xgb
)
rmse_ensemble = root_mean_squared_error(y_val_orig, y_pred_ensemble)
print(f'RMSE Ensemble ponderado: {rmse_ensemble:.2f}')

In [ ]:
# Resumen comparativo
results = pd.DataFrame({
    'Modelo': ['RandomForest', 'LightGBM', 'XGBoost', 'Ensemble'],
    'RMSE_val': [rmse_rf, rmse_lgb, rmse_xgb, rmse_ensemble]
}).sort_values('RMSE_val')
print(results.to_string(index=False))

results.set_index('Modelo')['RMSE_val'].plot(kind='barh', color='steelblue', figsize=(8, 3))
plt.xlabel('RMSE')
plt.title('Comparativa de modelos (validación)')
plt.tight_layout()
plt.show()

## 5. Reentrenamiento sobre todos los datos de `train.csv`

In [ ]:
# Feature engineering sobre todos los datos
X_fe_all = feature_engineering(X)
y_log_all = np.log1p(y)

# Preprocessor final (fit sobre todos los datos de train)
final_preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols)
], remainder='drop')

X_all_proc = final_preprocessor.fit_transform(X_fe_all)
print('Shape final procesado:', X_all_proc.shape)

In [ ]:
# Reentrenamos los 3 modelos con los mejores params sobre TODOS los datos
final_rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
final_rf.fit(X_all_proc, y_log_all)

final_lgb = HistGradientBoostingRegressor(**rs_hgb.best_params_, random_state=42)
final_lgb.fit(X_all_proc, y_log_all)

final_xgb = GradientBoostingRegressor(**rs_gb.best_params_, random_state=42)
final_xgb.fit(X_all_proc, y_log_all)

print("Modelos finales entrenados sobre 100% de train.csv")


---
# PARTE 2: Predicción y submission

## 6. Carga y procesado de `test.csv`

In [ ]:
X_pred = pd.read_csv('./data/test.csv', encoding='latin-1')
print('Shape test:', X_pred.shape)

X_pred_fe   = feature_engineering(X_pred)
X_pred_proc = final_preprocessor.transform(X_pred_fe)  # Solo transform, nunca fit
print('Shape test procesado:', X_pred_proc.shape)

## 7. Genera la submission

In [ ]:
# Predicciones individuales en escala original
pred_rf  = np.expm1(final_rf.predict(X_pred_proc))
pred_lgb = np.expm1(final_lgb.predict(X_pred_proc))
pred_xgb = np.expm1(final_xgb.predict(X_pred_proc))

# Ensemble ponderado (mismos pesos calculados en validación)
final_predictions = (
    weights[0] * pred_rf +
    weights[1] * pred_lgb +
    weights[2] * pred_xgb
)

submission = pd.DataFrame({
    'laptop_ID': X_pred['laptop_ID'],
    'Price_in_euros': final_predictions
})

submission.head(10)

## 8. Checker y guardado

In [ ]:
def checker(df_to_submit, sample, filename=None):
    if df_to_submit.shape != sample.shape:
        print('Shape incorrecto.')
        print(f'Tu submission: {df_to_submit.shape} | Esperado: {sample.shape}')
        return
    if not (df_to_submit.columns == sample.columns).all():
        print('Nombres de columnas incorrectos.')
        return
    if not (df_to_submit['laptop_ID'] == sample['laptop_ID']).all():
        print('Los IDs no coinciden con sample_submission.')
        return
    if filename is None:
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'submission_{timestamp}.csv'
    df_to_submit.to_csv(filename, index=False)
    print(f'✅ Submission guardada como: {filename}')
    return filename


sample = pd.read_csv('./data/sample_submission.csv', encoding='latin-1')
checker(submission, sample)

## 9. Análisis de errores (opcional)

In [ ]:
# Importancia de features según RandomForest
feat_names = (
    list(final_preprocessor.named_transformers_["cat"].get_feature_names_out(cat_cols)) +
    num_cols
)
importances = pd.Series(
    final_rf.feature_importances_,
    index=feat_names
).sort_values(ascending=False).head(20)

importances.plot(kind="barh", figsize=(8, 6), color="mediumpurple")
plt.title("Top 20 features importantes (RandomForest)")
plt.tight_layout()
plt.show()


In [ ]:
# Análisis residuos en validación
residuos = y_val_orig.values - y_pred_ensemble
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(y_pred_ensemble, residuos, alpha=0.4, color='steelblue')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicción'); plt.ylabel('Residuo'); plt.title('Residuos vs Predicción')
plt.subplot(1, 2, 2)
plt.hist(residuos, bins=40, color='tomato')
plt.xlabel('Residuo'); plt.title('Distribución de residuos')
plt.tight_layout()
plt.show()

print(f'RMSE final ensemble en validación: {rmse_ensemble:.2f}')